# 03. 피처 엔지니어링 전 날짜·요일 점검

정제된 일별 물량·이벤트 데이터를 대상으로 시계열 피처 생성 전에 필요한 날짜 구조를 확인한다.

이번 단계의 범위는 다음과 같다.

- `접수일자`와 기존 `요일` 값의 일치 여부 확인
- 분석 기간의 달력상 공백 날짜 확인
- 공백 날짜를 주말과 평일로 구분
- 실제 데이터에 포함된 주말 날짜 확인
- 이벤트 라벨 검증 및 평시·이벤트 데이터 분리
- 월·일·요일 원-핫·직전 관측일 간격 피처 생성
- 과거 관측 물량 기반 Lag·이동평균 피처 생성

> 이 노트북에서는 공백 날짜를 추가하거나 값을 0으로 채우지 않으며, Lag·이동평균 등의 피처도 생성하지 않는다.

## 1. 데이터 불러오기 및 기본 검증

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / "data" / "interim" / "daily_volume_event_clean.csv"

EXPECTED_COLUMNS = [
    "접수일자",
    "접수지역",
    "접수통수",
    "요일",
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
]

assert INPUT_PATH.exists(), f"입력 파일이 없습니다: {INPUT_PATH}"

daily = pd.read_csv(
    INPUT_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)

assert daily.columns.tolist() == EXPECTED_COLUMNS, "예상한 컬럼 구조와 다릅니다."
assert daily["접수일자"].notna().all(), "날짜 결측치가 있습니다."
assert daily["접수일자"].is_unique, "중복 날짜가 있습니다."
assert daily["접수일자"].is_monotonic_increasing, "날짜가 오름차순이 아닙니다."

basic_summary = pd.DataFrame(
    {
        "항목": ["입력 파일", "행 수", "열 수", "시작일", "종료일", "중복 날짜"],
        "결과": [
            str(INPUT_PATH.relative_to(PROJECT_ROOT)),
            f"{len(daily):,}",
            len(daily.columns),
            daily["접수일자"].min().date().isoformat(),
            daily["접수일자"].max().date().isoformat(),
            int(daily["접수일자"].duplicated().sum()),
        ],
    }
)

display(basic_summary)

,항목,결과
0,입력 파일,data/interim/daily_volume_event_clean.csv
1,행 수,617
2,열 수,11
3,시작일,2024-01-02
4,종료일,2026-06-30
5,중복 날짜,0


## 2. 접수일자와 요일 일치 여부

In [2]:
WEEKDAY_KO = {
    0: "월",
    1: "화",
    2: "수",
    3: "목",
    4: "금",
    5: "토",
    6: "일",
}

weekday_check = daily[["접수일자", "요일"]].copy()
weekday_check["계산요일"] = weekday_check["접수일자"].dt.dayofweek.map(WEEKDAY_KO)
weekday_check["일치여부"] = weekday_check["요일"].eq(weekday_check["계산요일"])

weekday_summary = pd.DataFrame(
    {
        "항목": ["전체 행", "요일 일치", "요일 불일치"],
        "건수": [
            len(weekday_check),
            int(weekday_check["일치여부"].sum()),
            int((~weekday_check["일치여부"]).sum()),
        ],
    }
)

display(weekday_summary)
display(weekday_check.loc[~weekday_check["일치여부"]])

assert weekday_check["일치여부"].all(), "접수일자와 요일이 불일치하는 행이 있습니다."


,항목,건수
0,전체 행,617
1,요일 일치,617
2,요일 불일치,0


,접수일자,요일,계산요일,일치여부


## 3. 달력상 공백 날짜 확인

In [3]:
calendar_dates = pd.date_range(
    start=daily["접수일자"].min(),
    end=daily["접수일자"].max(),
    freq="D",
)
observed_dates = pd.DatetimeIndex(daily["접수일자"])
missing_dates = calendar_dates.difference(observed_dates)

missing_date_detail = pd.DataFrame({"접수일자": missing_dates})
missing_date_detail["연도"] = missing_date_detail["접수일자"].dt.year
missing_date_detail["월"] = missing_date_detail["접수일자"].dt.month
missing_date_detail["요일"] = (
    missing_date_detail["접수일자"].dt.dayofweek.map(WEEKDAY_KO)
)
missing_date_detail["구분"] = (
    missing_date_detail["접수일자"]
    .dt.dayofweek.ge(5)
    .map({True: "주말", False: "평일"})
)

gap_summary = pd.DataFrame(
    {
        "항목": ["전체 달력일", "관측일", "공백일", "주말 공백", "평일 공백"],
        "건수": [
            len(calendar_dates),
            len(observed_dates),
            len(missing_date_detail),
            int(missing_date_detail["구분"].eq("주말").sum()),
            int(missing_date_detail["구분"].eq("평일").sum()),
        ],
    }
)

missing_weekday_counts = (
    missing_date_detail.loc[missing_date_detail["구분"].eq("평일"), "요일"]
    .value_counts()
    .reindex(["월", "화", "수", "목", "금"], fill_value=0)
    .rename_axis("요일")
    .reset_index(name="공백일 수")
)

display(gap_summary)
display(missing_weekday_counts)
display(missing_date_detail.loc[missing_date_detail["구분"].eq("평일")])

,항목,건수
0,전체 달력일,911
1,관측일,617
2,공백일,294
3,주말 공백,252
4,평일 공백,42


,요일,공백일 수
0,월,10
1,화,8
2,수,10
3,목,7
4,금,7


,접수일자,연도,월,요일,구분
10,2024-02-09,2024,2,금,평일
13,2024-02-12,2024,2,월,평일
18,2024-03-01,2024,3,금,평일
29,2024-04-10,2024,4,수,평일
38,2024-05-06,2024,5,월,평일
41,2024-05-15,2024,5,수,평일
48,2024-06-06,2024,6,목,평일
69,2024-08-15,2024,8,목,평일
80,2024-09-16,2024,9,월,평일
81,2024-09-17,2024,9,화,평일


## 4. 실제 데이터에 포함된 주말 확인

In [4]:
observed_weekends = daily.loc[
    daily["접수일자"].dt.dayofweek.ge(5),
    ["접수일자", "요일", "접수통수"],
].copy()

observed_weekend_summary = pd.DataFrame(
    {
        "항목": ["관측된 주말 날짜"],
        "건수": [len(observed_weekends)],
    }
)

display(observed_weekend_summary)
display(observed_weekends)

,항목,건수
0,관측된 주말 날짜,8


,접수일자,요일,접수통수
61,2024-03-30,토,678618
62,2024-03-31,일,4922
300,2025-03-22,토,49384
339,2025-05-18,일,438604
345,2025-05-24,토,255304
364,2025-06-22,일,65
590,2026-05-23,토,595408
591,2026-05-24,일,109723


## 5. 점검 결과

In [5]:
audit_result = pd.DataFrame(
    {
        "점검 항목": [
            "접수일자와 요일 불일치",
            "달력상 전체 공백일",
            "주말 공백일",
            "평일 공백일",
            "실제 데이터에 포함된 주말",
            "이번 단계의 데이터 변경",
        ],
        "결과": [
            int((~weekday_check["일치여부"]).sum()),
            len(missing_date_detail),
            int(missing_date_detail["구분"].eq("주말").sum()),
            int(missing_date_detail["구분"].eq("평일").sum()),
            len(observed_weekends),
            "없음",
        ],
    }
)

display(audit_result)

print("날짜·요일 사전 점검 완료")
print("- 공백 날짜 추가 또는 0 채움: 수행하지 않음")
print("- Lag·이동평균 등 피처 생성: 수행하지 않음")

,점검 항목,결과
0,접수일자와 요일 불일치,0
1,달력상 전체 공백일,294
2,주말 공백일,252
3,평일 공백일,42
4,실제 데이터에 포함된 주말,8
5,이번 단계의 데이터 변경,없음


날짜·요일 사전 점검 완료
- 공백 날짜 추가 또는 0 채움: 수행하지 않음
- Lag·이동평균 등 피처 생성: 수행하지 않음


## 6. 평시(Baseline) 데이터 분리

7개 세금·보험료 이벤트 컬럼이 모두 `0`이면 평시(`is_event = 0`), 하나 이상 `1`이면 이벤트 발생일(`is_event = 1`)로 구분한다.

이 단계에서는 분류 결과를 노트북 메모리에서만 확인하며 별도 파일로 저장하지 않는다.

In [6]:
EVENT_COLUMNS = [
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
]

event_value_check = pd.DataFrame(
    {
        "이벤트 컬럼": EVENT_COLUMNS,
        "고유값": [
            sorted(daily[column].dropna().unique().tolist())
            for column in EVENT_COLUMNS
        ],
    }
)

invalid_event_values = {
    column: sorted(set(daily[column].dropna().unique()) - {0, 1})
    for column in EVENT_COLUMNS
}
invalid_event_values = {
    column: values
    for column, values in invalid_event_values.items()
    if values
}

display(event_value_check)
assert not invalid_event_values, (
    f"0/1 이외의 이벤트 값이 있습니다: {invalid_event_values}"
)

,이벤트 컬럼,고유값
0,제1기분 자동차세,"[0, 1]"
1,재산세(건축),"[0, 1]"
2,정기분 주민세,"[0, 1]"
3,주민세(사업소분),"[0, 1]"
4,재산세(토지),"[0, 1]"
5,제2기분 자동차세,"[0, 1]"
6,사회보험료 통합,"[0, 1]"


In [7]:
modeling_base = daily.copy()
modeling_base["event_count"] = (
    modeling_base[EVENT_COLUMNS].sum(axis=1).astype("int8")
)
modeling_base["is_event"] = modeling_base["event_count"].gt(0).astype("int8")

baseline_data = modeling_base.loc[modeling_base["is_event"].eq(0)].copy()
event_data = modeling_base.loc[modeling_base["is_event"].eq(1)].copy()

assert len(baseline_data) + len(event_data) == len(modeling_base)
assert baseline_data[EVENT_COLUMNS].eq(0).all(axis=None)
assert event_data[EVENT_COLUMNS].eq(1).any(axis=1).all()
assert modeling_base["is_event"].isin([0, 1]).all()

split_summary = pd.DataFrame(
    {
        "구분": ["전체", "평시", "이벤트 발생일", "복수 이벤트 발생일"],
        "행 수": [
            len(modeling_base),
            len(baseline_data),
            len(event_data),
            int(modeling_base["event_count"].gt(1).sum()),
        ],
        "전체 대비 비율(%)": [
            100.0,
            round(len(baseline_data) / len(modeling_base) * 100, 1),
            round(len(event_data) / len(modeling_base) * 100, 1),
            round(
                modeling_base["event_count"].gt(1).sum()
                / len(modeling_base)
                * 100,
                1,
            ),
        ],
    }
)

period_summary = pd.DataFrame(
    {
        "구분": ["평시", "이벤트 발생일"],
        "시작일": [
            baseline_data["접수일자"].min().date().isoformat(),
            event_data["접수일자"].min().date().isoformat(),
        ],
        "종료일": [
            baseline_data["접수일자"].max().date().isoformat(),
            event_data["접수일자"].max().date().isoformat(),
        ],
    }
)

display(split_summary)
display(period_summary)

,구분,행 수,전체 대비 비율(%)
0,전체,617,100.0
1,평시,446,72.3
2,이벤트 발생일,171,27.7
3,복수 이벤트 발생일,2,0.3


,구분,시작일,종료일
0,평시,2024-01-02,2026-06-30
1,이벤트 발생일,2024-01-22,2026-06-25


In [8]:
event_label_summary = (
    modeling_base[EVENT_COLUMNS]
    .sum()
    .astype(int)
    .rename_axis("이벤트")
    .reset_index(name="활성 날짜 수")
    .sort_values(["활성 날짜 수", "이벤트"], ascending=[False, True])
    .reset_index(drop=True)
)

event_overlap_summary = (
    modeling_base["event_count"]
    .value_counts()
    .sort_index()
    .rename_axis("동시 활성 이벤트 수")
    .reset_index(name="날짜 수")
)

display(event_label_summary)
display(event_overlap_summary)

print("평시·이벤트 데이터 분리 완료")
print(f"- 평시: {len(baseline_data):,}일")
print(f"- 이벤트 발생일: {len(event_data):,}일")
print(
    "- 복수 이벤트 발생일: "
    f"{int(modeling_base['event_count'].gt(1).sum()):,}일"
)
print("- 원본 CSV 변경 또는 별도 파일 저장: 수행하지 않음")

,이벤트,활성 날짜 수
0,사회보험료 통합,108
1,재산세(건축),15
2,제1기분 자동차세,15
3,정기분 주민세,10
4,제2기분 자동차세,10
5,주민세(사업소분),10
6,재산세(토지),5


,동시 활성 이벤트 수,날짜 수
0,0,446
1,1,169
2,2,2


평시·이벤트 데이터 분리 완료
- 평시: 446일
- 이벤트 발생일: 171일
- 복수 이벤트 발생일: 2일
- 원본 CSV 변경 또는 별도 파일 저장: 수행하지 않음


## 7. 날짜 기반 피처 생성

승인된 범위에 따라 다음 피처만 생성한다.

- `month`: 접수 월
- `day`: 접수 일
- `weekday_월` ~ `weekday_일`: 요일 원-핫 인코딩
- `days_since_prev`: 직전 관측일과의 날짜 간격

첫 관측일은 비교할 이전 날짜가 없으므로 `days_since_prev`를 결측으로 유지한다. 이 단계에서는 Lag·이동평균을 만들거나 결과를 파일로 저장하지 않는다.

In [9]:
WEEKDAY_ONEHOT_COLUMNS = [
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
]

feature_data = modeling_base.copy()
columns_before_feature_creation = feature_data.columns.tolist()

feature_data["month"] = feature_data["접수일자"].dt.month.astype("int8")
feature_data["day"] = feature_data["접수일자"].dt.day.astype("int8")

weekday_onehot = pd.get_dummies(
    feature_data["요일"],
    prefix="weekday",
    prefix_sep="_",
    dtype="int8",
).reindex(columns=WEEKDAY_ONEHOT_COLUMNS, fill_value=0)

feature_data = pd.concat([feature_data, weekday_onehot], axis=1)
feature_data["days_since_prev"] = (
    feature_data["접수일자"].diff().dt.days.astype("Int16")
)

CREATED_CALENDAR_FEATURES = [
    "month",
    "day",
    *WEEKDAY_ONEHOT_COLUMNS,
    "days_since_prev",
]

assert len(feature_data) == len(modeling_base), "피처 생성 후 행 수가 달라졌습니다."
assert feature_data[columns_before_feature_creation].equals(modeling_base)
assert feature_data[WEEKDAY_ONEHOT_COLUMNS].sum(axis=1).eq(1).all()
assert feature_data["month"].between(1, 12).all()
assert feature_data["day"].between(1, 31).all()
assert feature_data["days_since_prev"].isna().sum() == 1
assert pd.isna(feature_data.loc[feature_data.index[0], "days_since_prev"])

feature_summary = pd.DataFrame(
    {
        "피처": CREATED_CALENDAR_FEATURES,
        "자료형": [
            str(feature_data[column].dtype)
            for column in CREATED_CALENDAR_FEATURES
        ],
        "결측치 수": [
            int(feature_data[column].isna().sum())
            for column in CREATED_CALENDAR_FEATURES
        ],
    }
)

display(feature_summary)

,피처,자료형,결측치 수
0,month,int8,0
1,day,int8,0
2,weekday_월,int8,0
3,weekday_화,int8,0
4,weekday_수,int8,0
5,weekday_목,int8,0
6,weekday_금,int8,0
7,weekday_토,int8,0
8,weekday_일,int8,0
9,days_since_prev,Int16,1


In [10]:
weekday_onehot_check = pd.concat(
    [
        feature_data[["접수일자", "요일"]],
        feature_data[WEEKDAY_ONEHOT_COLUMNS],
    ],
    axis=1,
)

gap_distribution = (
    feature_data["days_since_prev"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("직전 관측일과의 간격(일)")
    .reset_index(name="행 수")
)

calendar_feature_preview_columns = [
    "접수일자",
    "요일",
    "month",
    "day",
    *WEEKDAY_ONEHOT_COLUMNS,
    "days_since_prev",
]

display(weekday_onehot_check.head(10))
display(gap_distribution)
display(feature_data[calendar_feature_preview_columns].head(10))

print("날짜 기반 피처 생성 완료")
print(f"- 생성 피처 수: {len(CREATED_CALENDAR_FEATURES)}개")
print(f"- 피처 생성 후 행 수: {len(feature_data):,}행")
print(
    "- days_since_prev 결측치: "
    f"{int(feature_data['days_since_prev'].isna().sum())}건 "
    "(첫 관측일)"
)
print("- Lag·이동평균 생성 및 파일 저장: 수행하지 않음")

,접수일자,요일,weekday_월,weekday_화,weekday_수,weekday_목,weekday_금,weekday_토,weekday_일
0,2024-01-02,화,0,1,0,0,0,0,0
1,2024-01-03,수,0,0,1,0,0,0,0
2,2024-01-04,목,0,0,0,1,0,0,0
3,2024-01-05,금,0,0,0,0,1,0,0
4,2024-01-08,월,1,0,0,0,0,0,0
5,2024-01-09,화,0,1,0,0,0,0,0
6,2024-01-10,수,0,0,1,0,0,0,0
7,2024-01-11,목,0,0,0,1,0,0,0
8,2024-01-12,금,0,0,0,0,1,0,0
9,2024-01-15,월,1,0,0,0,0,0,0


,직전 관측일과의 간격(일),행 수
0,1,473
1,2,19
2,3,110
3,4,8
4,5,2
5,6,2
6,7,1
7,8,1
8,<NA>,1


,접수일자,요일,month,day,weekday_월,weekday_화,weekday_수,weekday_목,weekday_금,weekday_토,weekday_일,days_since_prev
0,2024-01-02,화,1,2,0,1,0,0,0,0,0,<NA>
1,2024-01-03,수,1,3,0,0,1,0,0,0,0,1
2,2024-01-04,목,1,4,0,0,0,1,0,0,0,1
3,2024-01-05,금,1,5,0,0,0,0,1,0,0,1
4,2024-01-08,월,1,8,1,0,0,0,0,0,0,3
5,2024-01-09,화,1,9,0,1,0,0,0,0,0,1
6,2024-01-10,수,1,10,0,0,1,0,0,0,0,1
7,2024-01-11,목,1,11,0,0,0,1,0,0,0,1
8,2024-01-12,금,1,12,0,0,0,0,1,0,0,1
9,2024-01-15,월,1,15,1,0,0,0,0,0,0,3


날짜 기반 피처 생성 완료
- 생성 피처 수: 10개
- 피처 생성 후 행 수: 617행
- days_since_prev 결측치: 1건 (첫 관측일)
- Lag·이동평균 생성 및 파일 저장: 수행하지 않음


## 8. Lag 및 이동평균 피처 생성

관측일 간격이 일정하지 않으므로 행 순서상 이전 관측일을 기준으로 과거 물량 피처를 생성한다.

- `lag_1`: 직전 1개 관측일의 접수통수
- `lag_5`: 직전 5개 관측일 전의 접수통수
- `rolling_mean_5`: 현재 행을 제외한 직전 5개 관측일의 평균 접수통수
- `rolling_mean_20`: 현재 행을 제외한 직전 20개 관측일의 평균 접수통수

이동평균은 `접수통수.shift(1)`을 먼저 적용한 뒤 계산하여 현재 행의 실제 물량이 입력 피처에 섞이지 않도록 한다. 선행 구간의 자연 결측치는 임의로 채우지 않는다.

In [11]:
PAST_VOLUME_FEATURES = [
    "lag_1",
    "lag_5",
    "rolling_mean_5",
    "rolling_mean_20",
]

feature_data = feature_data.copy()
target_volume = feature_data["접수통수"]
past_volume = target_volume.shift(1)

feature_data["lag_1"] = target_volume.shift(1)
feature_data["lag_5"] = target_volume.shift(5)
feature_data["rolling_mean_5"] = past_volume.rolling(
    window=5,
    min_periods=5,
).mean()
feature_data["rolling_mean_20"] = past_volume.rolling(
    window=20,
    min_periods=20,
).mean()

expected_missing_counts = {
    "lag_1": 1,
    "lag_5": 5,
    "rolling_mean_5": 5,
    "rolling_mean_20": 20,
}

actual_missing_counts = {
    column: int(feature_data[column].isna().sum())
    for column in PAST_VOLUME_FEATURES
}

assert actual_missing_counts == expected_missing_counts
pd.testing.assert_series_equal(
    feature_data["lag_1"].iloc[1:].reset_index(drop=True),
    target_volume.iloc[:-1].reset_index(drop=True),
    check_dtype=False,
    check_names=False,
)
pd.testing.assert_series_equal(
    feature_data["lag_5"].iloc[5:].reset_index(drop=True),
    target_volume.iloc[:-5].reset_index(drop=True),
    check_dtype=False,
    check_names=False,
)
assert feature_data.loc[feature_data.index[5], "rolling_mean_5"] == (
    target_volume.iloc[:5].mean()
)
assert feature_data.loc[feature_data.index[20], "rolling_mean_20"] == (
    target_volume.iloc[:20].mean()
)

past_feature_summary = pd.DataFrame(
    {
        "피처": PAST_VOLUME_FEATURES,
        "기준": [
            "직전 1개 관측일",
            "직전 5개 관측일 전",
            "직전 5개 관측일 평균",
            "직전 20개 관측일 평균",
        ],
        "결측치 수": [
            actual_missing_counts[column]
            for column in PAST_VOLUME_FEATURES
        ],
    }
)

display(past_feature_summary)

,피처,기준,결측치 수
0,lag_1,직전 1개 관측일,1
1,lag_5,직전 5개 관측일 전,5
2,rolling_mean_5,직전 5개 관측일 평균,5
3,rolling_mean_20,직전 20개 관측일 평균,20


In [12]:
past_feature_preview_columns = [
    "접수일자",
    "접수통수",
    *PAST_VOLUME_FEATURES,
]

past_feature_correlation = (
    feature_data[PAST_VOLUME_FEATURES]
    .corrwith(feature_data["접수통수"])
    .round(4)
    .rename_axis("피처")
    .reset_index(name="접수통수와의 상관계수")
)

display(feature_data[past_feature_preview_columns].head(25))
display(past_feature_correlation)

print("Lag·이동평균 피처 생성 완료")
print(f"- 생성 피처 수: {len(PAST_VOLUME_FEATURES)}개")
print("- 현재 행의 접수통수 사용: 없음")
print("- 선행 구간 결측치 대체: 수행하지 않음")
print("- Train/Test 분할 및 파일 저장: 수행하지 않음")

,접수일자,접수통수,lag_1,lag_5,rolling_mean_5,rolling_mean_20
0,2024-01-02,85267,NaN,NaN,NaN,NaN
1,2024-01-03,118718,85267.0,NaN,NaN,NaN
2,2024-01-04,93382,118718.0,NaN,NaN,NaN
3,2024-01-05,57788,93382.0,NaN,NaN,NaN
4,2024-01-08,81652,57788.0,NaN,NaN,NaN
5,2024-01-09,52864,81652.0,85267.0,87361.4,NaN
6,2024-01-10,79009,52864.0,118718.0,80880.8,NaN
7,2024-01-11,281812,79009.0,93382.0,72939.0,NaN
8,2024-01-12,139924,281812.0,57788.0,110625.0,NaN
9,2024-01-15,87097,139924.0,81652.0,127052.2,NaN


,피처,접수통수와의 상관계수
0,lag_1,0.1384
1,lag_5,0.0134
2,rolling_mean_5,0.0394
3,rolling_mean_20,-0.0063


Lag·이동평균 피처 생성 완료
- 생성 피처 수: 4개
- 현재 행의 접수통수 사용: 없음
- 선행 구간 결측치 대체: 수행하지 않음
- Train/Test 분할 및 파일 저장: 수행하지 않음
